# Member 2 — Optimal Configuration & Cross-Member Analysis
## CS690R — IRB Stroke Recovery Dataset

This notebook has two parts:

**Part 1 — Optimal Configuration**
Runs the best setup discovered from the ablation study: top 5 Y-axis features + 100 trees.
Also provides clinical interpretations of each of the 5 features.

**Part 2 — Cross-Member Combined Figure**
Puts Member 2's feature importances (RF) side by side with Member 3's Spearman
correlations to visually tell the Y-axis vs Z-axis story in one figure.

---
## 0 — Imports and Setup

In [ ]:
import subprocess, sys
for pkg in ['scikit-learn', 'seaborn', 'matplotlib', 'pandas', 'numpy']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')

FEATURES_PATH = 'features/biopm_features.npz'
FIGURES_DIR   = 'results/figures'
METRICS_DIR   = 'results/metrics'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size'  : 11,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

BASE_AUC      = 0.780   # all 228 features, 200 trees
GROUP_COLORS  = {0: '#2196F3', 1: '#F44336'}
GROUP_LABELS  = {0: 'Healthy', 1: 'Stroke'}
print('Imports OK')

---
## 1 — Load Data

In [ ]:
d = np.load(FEATURES_PATH, allow_pickle=True)
X             = d['features'].astype(np.float64)
y             = d['labels'].astype(int)
pids          = d['pids'].astype(int)
feature_names = list(d['feature_names'])

# Load pre-ranked importances from base RF run
importance_df = pd.read_csv(os.path.join(METRICS_DIR, 'rf_feature_importances_irb.csv'))

# Top 5 features
TOP5_NAMES   = importance_df['feature'].head(5).tolist()
TOP5_INDICES = [feature_names.index(f) for f in TOP5_NAMES]
X_top5       = X[:, TOP5_INDICES]

print('Top 5 features:')
for i, (name, idx) in enumerate(zip(TOP5_NAMES, TOP5_INDICES)):
    print(f'  {i+1}. {name}  (importance={importance_df.iloc[i]["importance"]:.4f})')

---
# PART 1 — Optimal Configuration
## 2 — Clinical Interpretation of Top 5 Features

Before running the model, let's understand what each feature actually measures clinically.
All 5 features are Y-axis (vertical arm motion) features — this is the axis most affected by stroke.

In [ ]:
# Clinical interpretations of the top 5 features
clinical_notes = {
    'Y_acc_std_p10': (
        'Y-axis Acceleration Std — 10th Percentile',
        'Measures the minimum variability in vertical arm acceleration across the day. '
        'Healthy people maintain a consistent baseline of dynamic arm movement even at '
        'their least active moments. Stroke patients show much lower minimum variability, '
        'meaning even at their most active, their vertical movement is less dynamic. '
        'Clinical meaning: captures the floor of arm movement quality.'
    ),
    'Y_jerk_std_p10': (
        'Y-axis Jerk Std — 10th Percentile',
        'Jerk is the rate of change of acceleration — it captures movement smoothness. '
        'High jerk std means movement changes speed rapidly (dynamic, coordinated). '
        'Low jerk std means slow, hesitant, or rigid movement. The p10 captures the '
        'minimum smoothness across the day. Stroke survivors show consistently lower '
        'jerk variability, indicating reduced movement coordination. '
        'Clinical meaning: proxy for motor coordination and fluidity.'
    ),
    'Y_acc_mean_p90': (
        'Y-axis Acceleration Mean — 90th Percentile',
        'The 90th percentile of mean vertical acceleration captures peak arm activity '
        'levels. Healthy people reach higher vertical acceleration peaks during daily '
        'activities (reaching, lifting, gesturing). Stroke patients have a lower ceiling '
        'on their best vertical movement performance. '
        'Clinical meaning: upper bound of functional arm use capacity.'
    ),
    'Y_acc_mean_p10': (
        'Y-axis Acceleration Mean — 10th Percentile',
        'The 10th percentile of mean vertical acceleration is typically negative '
        '(downward motion). Healthy people show a consistent negative baseline reflecting '
        'natural downward arm swing. Stroke patients show a wider and less consistent '
        'distribution, reflecting asymmetric or gravity-dominated arm positioning. '
        'Clinical meaning: captures resting arm posture and passive gravitational loading.'
    ),
    'Y_vel_std_p90': (
        'Y-axis Velocity Std — 90th Percentile',
        'The 90th percentile of vertical velocity variability captures peak movement '
        'speed diversity. Healthy people show high velocity variability at peak activity, '
        'meaning they perform a wide range of fast and slow vertical movements. Stroke '
        'patients show much tighter distributions — their fastest movements are still '
        'slower and less varied than healthy controls. '
        'Clinical meaning: range of deliberate vertical movement speed.'
    ),
}

print('Clinical Interpretation of Top 5 Biomarkers')
print('=' * 65)
for rank, name in enumerate(TOP5_NAMES, 1):
    title, desc = clinical_notes[name]
    print(f'\n#{rank} {name}')
    print(f'    Full name : {title}')
    print(f'    Meaning   : {desc}')

---
## 3 — Run Optimal Configuration LOSO

In [ ]:
logo      = LeaveOneGroupOut()
all_probs = np.zeros(len(y))
all_preds = np.zeros(len(y), dtype=int)

print('Running optimal config: top 5 features, 100 trees...')

for train_idx, test_idx in logo.split(X_top5, y, pids):
    X_train, X_test = X_top5[train_idx], X_top5[test_idx]
    y_train         = y[train_idx]

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    all_probs[test_idx] = rf.predict_proba(X_test)[:, 1]
    all_preds[test_idx] = rf.predict(X_test)

# Find optimal threshold (maximises TPR - FPR)
fpr_arr, tpr_arr, thresholds = roc_curve(y, all_probs)
optimal_idx       = np.argmax(tpr_arr - fpr_arr)
optimal_threshold = thresholds[optimal_idx]
all_preds_opt     = (all_probs >= optimal_threshold).astype(int)

opt_auc = roc_auc_score(y, all_probs)
opt_f1  = f1_score(y, all_preds_opt, zero_division=0)

print(f'\n{"="*50}')
print(f'  Optimal Configuration Results')
print(f'{"="*50}')
print(f'  Features       : Top 5 Y-axis features')
print(f'  N Trees        : 100')
print(f'  Threshold      : {optimal_threshold:.3f} (maximises TPR-FPR)')
print(f'  Pooled AUC     : {opt_auc:.3f}  (baseline: {BASE_AUC})')
print(f'  Pooled F1      : {opt_f1:.3f}')
print(f'  AUC improvement: +{opt_auc - BASE_AUC:.3f}')
print(f'{"="*50}')

# Save
pd.DataFrame({
    'metric': ['pooled_auc', 'pooled_f1', 'optimal_threshold', 'n_features', 'n_trees', 'vs_baseline'],
    'value' : [opt_auc, opt_f1, optimal_threshold, 5, 100, opt_auc - BASE_AUC]
}).to_csv(os.path.join(METRICS_DIR, 'optimal_config_results_irb.csv'), index=False)
print('Saved → results/metrics/optimal_config_results_irb.csv')

---
## 4 — ROC Curve + Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── ROC Curve ────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(fpr_arr, tpr_arr, color='#2196F3', linewidth=2.5,
        label=f'Optimal config  (AUC = {opt_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random chance (AUC = 0.5)')

# Mark optimal threshold point
ax.scatter(fpr_arr[optimal_idx], tpr_arr[optimal_idx],
           color='#F44336', s=100, zorder=5,
           label=f'Optimal threshold ({optimal_threshold:.2f})')

# Shade AUC area
ax.fill_between(fpr_arr, tpr_arr, alpha=0.08, color='#2196F3')

# Annotate baseline AUC for comparison
ax.axhline(BASE_AUC, color='gray', linestyle=':', linewidth=1,
           label=f'Baseline AUC ({BASE_AUC})')

ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Optimal Configuration\n(Top 5 Y-axis Features, 100 Trees)',
             fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# ── Confusion Matrix ─────────────────────────────────────────────────────────
ax = axes[1]
cm = confusion_matrix(y, all_preds_opt)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Healthy', 'Stroke']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Optimal Configuration\n(at optimal threshold)',
             fontweight='bold')

# Annotate with rates
tn, fp, fn, tp = cm.ravel()
total_healthy = tn + fp
total_stroke  = fn + tp
ax.text(0.5, -0.18,
        f'Healthy correctly identified: {tn}/{total_healthy} ({100*tn/total_healthy:.0f}%)   '
        f'Stroke correctly identified: {tp}/{total_stroke} ({100*tp/total_stroke:.0f}%)',
        transform=ax.transAxes, ha='center', fontsize=9, color='#333333')

plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'optimal_config_roc_cm_irb.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

---
## 5 — Top 5 Feature Violin Plots (Optimal Config)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
palette = {GROUP_LABELS[0]: GROUP_COLORS[0], GROUP_LABELS[1]: GROUP_COLORS[1]}

for i, (feat_name, feat_idx) in enumerate(zip(TOP5_NAMES, TOP5_INDICES)):
    ax = axes[i]
    rows = [{'group': GROUP_LABELS[label], 'value': val}
            for val, label in zip(X[:, feat_idx], y)]
    sub = pd.DataFrame(rows)

    sns.violinplot(data=sub, x='group', y='value',
                   palette=palette, inner='box',
                   order=['Healthy', 'Stroke'],
                   ax=ax, linewidth=1.2)
    sns.stripplot(data=sub, x='group', y='value',
                  order=['Healthy', 'Stroke'],
                  color='black', alpha=0.25, size=2.5, jitter=True, ax=ax)

    title, _ = clinical_notes[feat_name]
    ax.set_title(f'#{i+1}\n{feat_name}', fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Feature value' if i == 0 else '', fontsize=9)
    ax.tick_params(labelsize=8)

legend_patches = [
    mpatches.Patch(color=GROUP_COLORS[0], label='Healthy'),
    mpatches.Patch(color=GROUP_COLORS[1], label='Stroke'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=2,
           fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.05))
fig.suptitle('Top 5 Optimal Biomarkers — Healthy vs. Stroke\nAll Y-axis features',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
out = os.path.join(FIGURES_DIR, 'optimal_config_violins_irb.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

---
# PART 2 — Cross-Member Combined Figure
## 6 — Load Member 3's Spearman Results

Member 3's `analysis.ipynb` saved clinical correlations to `results/metrics/clinical_correlations.csv`.
We load this alongside Member 2's feature importances to create the combined figure.

In [ ]:
# Load Member 3's Spearman correlations
corr_path = os.path.join(METRICS_DIR, 'clinical_correlations.csv')

if not os.path.exists(corr_path):
    print(f'ERROR: {corr_path} not found.')
    print('Make sure Member 3 has run analysis.ipynb and pushed their results.')
    print('Expected columns: feature, rho_ARAT, p_ARAT, rho_FMA, p_FMA')
else:
    corr_df = pd.read_csv(corr_path)
    corr_df['abs_rho_ARAT'] = corr_df['rho_ARAT'].abs()
    corr_df = corr_df.sort_values('abs_rho_ARAT', ascending=False).reset_index(drop=True)
    print('Member 3 Spearman results loaded successfully.')
    print(f'Total features with correlations: {len(corr_df)}')
    print('\nTop 10 by |ρ_ARAT|:')
    print(corr_df.head(10)[['feature', 'rho_ARAT', 'rho_FMA']].to_string(index=False))

---
## 7 — Combined Figure: RF Importance vs Spearman Correlation

This figure tells the core story of the project:
- **Left panel:** Top features for detecting stroke (Member 2) → Y-axis dominant
- **Right panel:** Top features for tracking recovery severity (Member 3) → Z-axis dominant

Two different clinical questions → two different feature sets.

In [ ]:
if not os.path.exists(corr_path):
    print('Skipping combined figure — Member 3 correlations not found.')
else:
    TOP_N = 15

    imp_top  = importance_df.head(TOP_N).iloc[::-1].copy()
    corr_top = corr_df.head(TOP_N).iloc[::-1].copy()

    def axis_color(name):
        """Color bars by which axis the feature belongs to."""
        if name.startswith('Y_'):       return '#2196F3'   # blue
        elif name.startswith('Z_'):     return '#4CAF50'   # green
        elif name.startswith('X_'):     return '#FF9800'   # orange
        else:                           return '#9C27B0'   # purple (magnitude)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # ── Left: RF Feature Importances (Member 2) ───────────────────────────────
    ax = axes[0]
    colors_imp = [axis_color(n) for n in imp_top['feature']]
    bars = ax.barh(imp_top['feature'], imp_top['importance'],
                   color=colors_imp, height=0.65,
                   xerr=imp_top['std'], ecolor='#555', capsize=2)
    for bar, val in zip(bars, imp_top['importance']):
        ax.text(bar.get_width() + 0.0003,
                bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_xlabel('Mean Decrease in Impurity\n(averaged across LOSO folds)', fontsize=10)
    ax.set_title(
        'Member 2 — RF Feature Importance\n"Which features detect stroke?"',
        fontweight='bold', fontsize=11, pad=10
    )
    ax.tick_params(axis='y', labelsize=8.5)
    ax.set_xlim(0, imp_top['importance'].max() * 1.22)

    # ── Right: Spearman Correlations (Member 3) ───────────────────────────────
    ax = axes[1]
    colors_corr = [axis_color(n) for n in corr_top['feature']]
    bars = ax.barh(corr_top['feature'], corr_top['abs_rho_ARAT'],
                   color=colors_corr, height=0.65)
    for bar, val, rho in zip(bars, corr_top['abs_rho_ARAT'], corr_top['rho_ARAT']):
        sign = '+' if rho >= 0 else '-'
        ax.text(bar.get_width() + 0.005,
                bar.get_y() + bar.get_height()/2,
                f'{sign}{val:.3f}', va='center', fontsize=8)
    ax.set_xlabel('|Spearman ρ| with ARAT score\n(higher = stronger clinical correlation)', fontsize=10)
    ax.set_title(
        'Member 3 — Spearman Correlation\n"Which features track recovery severity?"',
        fontweight='bold', fontsize=11, pad=10
    )
    ax.tick_params(axis='y', labelsize=8.5)
    ax.set_xlim(0, 1.05)

    # ── Shared legend ─────────────────────────────────────────────────────────
    legend_patches = [
        mpatches.Patch(color='#2196F3', label='Y-axis (vertical arm motion)'),
        mpatches.Patch(color='#4CAF50', label='Z-axis (forward-backward motion)'),
        mpatches.Patch(color='#FF9800', label='X-axis (side-to-side motion)'),
        mpatches.Patch(color='#9C27B0', label='Magnitude (acc_mag)'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4,
               fontsize=9.5, frameon=True, bbox_to_anchor=(0.5, -0.06),
               title='Feature axis', title_fontsize=10)

    # ── Annotation arrow / text ───────────────────────────────────────────────
    fig.text(0.5, 1.01,
             'Y-axis features detect stroke    ←    Different clinical questions    →    Z-axis features track recovery',
             ha='center', fontsize=11, color='#333333', style='italic')

    fig.suptitle(
        'Cross-Member Analysis — IRB Stroke Recovery Dataset\n'
        'Member 2 (RF Importance) vs Member 3 (Spearman Correlation)',
        fontsize=13, fontweight='bold', y=1.08
    )

    plt.tight_layout()
    out = os.path.join(FIGURES_DIR, 'cross_member_importance_vs_correlation_irb.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

---
## 8 — Summary

In [ ]:
print('=' * 60)
print('  PART 1 — Optimal Configuration')
print('=' * 60)
print(f'  Baseline  : 228 features, 200 trees → AUC {BASE_AUC:.3f}')
print(f'  Optimal   : 5 features,  100 trees → AUC {opt_auc:.3f}')
print(f'  Improvement: +{opt_auc - BASE_AUC:.3f} AUC')
print()
print('  Top 5 features (all Y-axis):')
for i, name in enumerate(TOP5_NAMES, 1):
    title, _ = clinical_notes[name]
    print(f'    {i}. {name} — {title}')

print()
print('=' * 60)
print('  PART 2 — Cross-Member Finding')
print('=' * 60)
print('  Detection (Member 2)  → Y-axis features dominate')
print('  Recovery  (Member 3)  → Z-axis features dominate')
print()
print('  Clinical interpretation:')
print('    Y-axis: best for screening/diagnosis (is this person impaired?)')
print('    Z-axis: best for monitoring/rehabilitation (how impaired are they?)')

print()
print('Output files:')
expected = [
    (os.path.join(METRICS_DIR, 'optimal_config_results_irb.csv'),              'Optimal config metrics'),
    (os.path.join(FIGURES_DIR, 'optimal_config_roc_cm_irb.png'),               'ROC curve + confusion matrix'),
    (os.path.join(FIGURES_DIR, 'optimal_config_violins_irb.png'),              'Top 5 violin plots'),
    (os.path.join(FIGURES_DIR, 'cross_member_importance_vs_correlation_irb.png'), 'Cross-member combined figure'),
]
for path, desc in expected:
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {path}  —  {desc}')